In [ ]:
import numpy as np
import pandas as pd

# -----------------------------
# 1. Create labeled dataset
# -----------------------------
df = pd.DataFrame({
    "user": ["A", "A", "B", "B", "C", "C"],  
    "item": ["Movie1", "Movie2", "Movie1", "Movie3", "Movie2", "Movie3"],
    "rating": [5, 3, 4, 2, 4, 1]
})

print("Original DataFrame:")
print(df)

# -----------------------------
# 2. Create ID mappings
# -----------------------------
user_ids = {u: i for i, u in enumerate(df["user"].unique())}
item_ids = {m: i for i, m in enumerate(df["item"].unique())}

n_users = len(user_ids)
n_items = len(item_ids)
k = 2  # embedding dimension

# -----------------------------
# 3. Initialize embeddings
# -----------------------------
np.random.seed(42)
user_emb = np.random.normal(0, 0.1, (n_users, k))
item_emb = np.random.normal(0, 0.1, (n_items, k))

# -----------------------------
# 4. Training loop (SGD)
# -----------------------------
lr = 0.05
reg = 0.01
epochs = 500

for epoch in range(epochs):
    total_loss = 0
    
    for _, row in df.iterrows():
        u = user_ids[row["user"]]
        i = item_ids[row["item"]]
        r = row["rating"]
        
        # prediction
        pred = user_emb[u].dot(item_emb[i])
        error = r - pred
        
        total_loss += error**2
        
        # gradient updates
        user_grad = error * item_emb[i] - reg * user_emb[u]
        item_grad = error * user_emb[u] - reg * item_emb[i]
        
        user_emb[u] += lr * user_grad
        item_emb[i] += lr * item_grad
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

# -----------------------------
# 5. Build full prediction matrix
# -----------------------------
pred_matrix = user_emb @ item_emb.T

pred_df = pd.DataFrame(
    pred_matrix,
    index=user_ids.keys(),
    columns=item_ids.keys()
)

print("\nPredicted Ratings Matrix:")
print(pred_df.round(2))

Original DataFrame:
  user    item  rating
0    A  Movie1       5
1    A  Movie2       3
2    B  Movie1       4
3    B  Movie3       2
4    C  Movie2       4
5    C  Movie3       1
Epoch 0, Loss: 70.8292
Epoch 100, Loss: 0.0050
Epoch 200, Loss: 0.0011
Epoch 300, Loss: 0.0011
Epoch 400, Loss: 0.0011

Predicted Ratings Matrix:
   Movie1  Movie2  Movie3
A    4.98    3.00    2.34
B    3.99    2.21    1.98
C    3.97    3.98    1.00


In [2]:
df

,user,item,rating
0,A,Movie1,5
1,A,Movie2,3
2,B,Movie1,4
3,B,Movie3,2
4,C,Movie2,4
5,C,Movie3,1
